# Question
        Farhad is a strange person who has n pet tigers and n² cages. Each bear i has a positive age aᵢ and a size sᵢ (no two tigers have the same age or size). Each cage j has a positive capacity cⱼ 
        and a specific distance dⱼ from Farhad’s bedroom (no two cages have the same capacity or the same distance). Farhad must place each bear in a separate cage.

Conditions for placing the tigers in cages:

        Farhad likes older tigers more and wants them to be placed closer to his bedroom. That means:
        If bear x is younger than bear y, then bear x must be placed in a cage that is farther from the cage of bear y.
        If a bear is placed in cage j with capacity cⱼ, and sᵢ > cⱼ, then the discomfort of the bear will be equal to sᵢ - cⱼ. Otherwise, the bear will not be uncomfortable.

Provide an O(n³) algorithm that assigns tigers to cages in a way that the above constraint is satisfied and the total discomfort of the tigers is minimized.        

# Answer
We have:

        * n tigers: [(a₁, s₁), ..., (aₙ, sₙ)] (age, size) sorted by ages.
        * n² cages: [(c₁, d₁), ..., (cₙ², dₙ²)] (capacity, distance) sorted by distance.
        * discomfort = max(0, sᵢ - cⱼ) if bear i is placed in cage j.


Idea:

        * dp[i][j] = minimum discomfort when we assign the first i tigers to the first j cages.
        * dp[1][j] for any j: Assign the oldest tiger (tiger 1) to cage j. Discomfort is max(s₁ - cⱼ, 0).
        * For i > 1, to assign the i-th tiger to the j-th cage, the (i-1)-th tiger must be assigned to some 
            cage k where k < j; so dp[i][j] = min(dp[i-1][k] + max(sⱼ - cⱼ, 0)) for all k < j.
        * The minimum total discomfort is the minimum value in dp[n][j] for j from n to n².


Complexity Analysis

        * Number of tigers: n.
        * Number of cages: n².
        * For each tiger i (from 1 to n), and for each possible cage j (from i to n² - (n - i)), we look at all previous cages k (from i-1 to j-1).
        * The total number of states is O(n * n²) = O(n³), and for each state, we do O(n) work in the worst case, 
            however we can precompute the prefix minima for dp[i-1][k] up to j-1, so that finding the minimum is O(1).
        * Thus, the total complexity becomes O(n³).

In [1]:
def min_discomfort(tigers, cages):
    n = len(tigers)
    # Sort tigers by age descending
    tigers.sort(key=lambda x: -x.age)
    # Sort cages by distance ascending
    cages.sort(key=lambda x: x.distance)
    
    # Initialize DP table
    dp = [[float('inf')] * (n*n + 1) for _ in range(n + 1)]
    dp[0][0] = 0
    
    for i in range(1, n + 1):
        min_prev = float('inf')
        # The i-th tiger can be placed in cages from i to n² - (n - i)
        for j in range(i, n*n - (n - i) + 1):
            # Update min_prev to be the min of dp[i-1][i-1 ... j-1]
            if j - 1 >= i - 1:
                min_prev = min(min_prev, dp[i-1][j-1])
            # Compute discomfort for tiger i in cage j
            discomfort = max(tigers[i-1].size - cages[j-1].capacity, 0)
            dp[i][j] = min_prev + discomfort
    
    # The answer is the min in dp[n][n ... n*n]
    return min(dp[n][n: n*n + 1])
    